In [10]:
# train_model.py
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
# merge_dataset.py
import pandas as pd
import glob

In [ ]:
# train_model.py

import pandas as pd
import glob
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit

# =========================
# Feature list
# =========================
FEATURES = [
    "dest_port", "window_duration",
    "fwd_packet_rate", "fwd_byte_rate",
    "bwd_packet_rate", "bwd_byte_rate",
    "pkt_len_mean", "pkt_len_std",
    "pkt_len_min", "pkt_len_max",
    "syn_ratio", "fin_ratio", "ack_ratio",
    "rst_ratio", "psh_ratio",
    "fwd_bwd_ratio", "byte_ratio",
    "iat_mean", "iat_std", "iat_min"
]

# =========================
# Load dataset files
# =========================
files = sorted(glob.glob("dataset_*.csv"))

if len(files) == 0:
    raise ValueError("No dataset files found")

print(f"[+] Found {len(files)} dataset files")

# =========================
# Load and clean all data
# =========================
dfs = []
for f in files:
    df = pd.read_csv(f)
    df.dropna(inplace=True)
    df["source_file"] = f
    dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True)

# Extract base flow ID (strips _w0, _w1 etc.) for grouping
full_df["base_flow_id"] = full_df["flow_id"].str.replace(r"_w\d+$", "", regex=True)

print(f"[+] Total rows before dedup check: {len(full_df)}")
print(f"[+] Duplicate feature rows: {full_df.duplicated(subset=FEATURES).sum()}")
print(f"[+] Unique windows (flow_id): {full_df['flow_id'].nunique()}")
print(f"[+] Unique flows (base_flow_id): {full_df['base_flow_id'].nunique()}")
print(f"\n[+] Overall label distribution:")
print(full_df["label"].value_counts())

# =========================
# Drop exact duplicates
# =========================
full_df = full_df.drop_duplicates(subset=FEATURES).reset_index(drop=True)
print(f"\n[+] Rows after dedup: {len(full_df)}")

# =========================
# Split by BASE FLOW ID
# Ensures all windows from the same flow stay on the same side
# of the train/test boundary — eliminates leakage
# =========================
groups = full_df["base_flow_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(full_df, groups=groups))

train_df = full_df.iloc[train_idx].reset_index(drop=True)
test_df  = full_df.iloc[test_idx].reset_index(drop=True)

# Shuffle both splits
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

# =========================
# Leakage verification — must be 0
# =========================
train_flows = set(train_df["base_flow_id"])
test_flows  = set(test_df["base_flow_id"])
leaked = train_flows & test_flows
print(f"\n[+] Flow leakage check: {len(leaked)} shared base flow_ids (must be 0)")
if leaked:
    raise RuntimeError(f"Leakage detected in flows: {leaked}")

print("\n[+] Train distribution:")
print(train_df["label"].value_counts())
print("\n[+] Test distribution:")
print(test_df["label"].value_counts())

# =========================
# Build feature matrices
# =========================
X_train = train_df[FEATURES].values
y_train = (train_df["label"] == "malicious").astype(int).values

X_test = test_df[FEATURES].values
y_test = (test_df["label"] == "malicious").astype(int).values

# Sanity check for NaN / inf
assert not np.isnan(X_train).any(), "NaN in X_train"
assert not np.isinf(X_train).any(), "Inf in X_train"
assert not np.isnan(X_test).any(),  "NaN in X_test"
assert not np.isinf(X_test).any(),  "Inf in X_test"

# =========================
# Train model
# =========================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\n[+] Training model...")
model.fit(X_train, y_train)

# =========================
# Evaluation
# =========================
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print("\n[+] Classification Report:")
print(classification_report(y_test, y_pred, target_names=["benign", "malicious"]))

print("\n[+] Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\n[+] Key Metrics:")
print(f"  True Positives  (attacks caught)  : {tp}")
print(f"  True Negatives  (benign correct)  : {tn}")
print(f"  False Positives (benign blocked)  : {fp}")
print(f"  False Negatives (attacks missed)  : {fn}")

fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
print(f"\n  False Positive Rate (FPR): {fpr:.4f}  <- aim < 0.05")
print(f"  False Negative Rate (FNR): {fnr:.4f}  <- aim < 0.10")

# =========================
# Probability distribution check
# =========================
print("\n[+] Predicted probability stats (malicious class):")
print(f"  mean : {y_pred_prob.mean():.4f}")
print(f"  std  : {y_pred_prob.std():.4f}")
print(f"  min  : {y_pred_prob.min():.4f}")
print(f"  max  : {y_pred_prob.max():.4f}")

thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
print("\n[+] Threshold sensitivity (adjust MALICIOUS_THRESHOLD in firewall.py):")
print(f"  {'Threshold':>10}  {'FPR':>8}  {'FNR':>8}  {'F1':>8}")
for t in thresholds:
    y_t      = (y_pred_prob >= t).astype(int)
    cm_t     = confusion_matrix(y_test, y_t, labels=[0, 1])
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    fpr_t    = fp_t / (fp_t + tn_t) if (fp_t + tn_t) > 0 else 0
    fnr_t    = fn_t / (fn_t + tp_t) if (fn_t + tp_t) > 0 else 0
    f1_t     = f1_score(y_test, y_t, zero_division=0)
    print(f"  {t:>10.1f}  {fpr_t:>8.4f}  {fnr_t:>8.4f}  {f1_t:>8.4f}")

# =========================
# Feature importance
# =========================
print("\n[+] Feature importances (descending):")
importances = sorted(
    zip(FEATURES, model.feature_importances_),
    key=lambda x: x[1], reverse=True
)


In [19]:
# train_model.py

import pandas as pd
import glob
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# =========================
# Load dataset files
# =========================
files = sorted(glob.glob("dataset_*.csv"))

if len(files) == 0:
    raise ValueError("No dataset files found")

print(f"[+] Found {len(files)} dataset files")

# =========================
# Load and clean all data
# =========================
dfs = []
for f in files:
    df = pd.read_csv(f)
    df.dropna(inplace=True)
    df["source_file"] = f  # track origin
    dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True)

print(f"[+] Total rows: {len(full_df)}")
print("\n[+] Overall label distribution:")
print(full_df["label"].value_counts())

# =========================
# Split EACH CLASS separately
# =========================
train_parts = []
test_parts = []

for label in full_df["label"].unique():
    class_df = full_df[full_df["label"] == label].sample(frac=1, random_state=42)

    split_idx = int(len(class_df) * 0.8)

    train_parts.append(class_df.iloc[:split_idx])
    test_parts.append(class_df.iloc[split_idx:])

train_df = pd.concat(train_parts, ignore_index=True)
test_df  = pd.concat(test_parts, ignore_index=True)

# Shuffle
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n[+] Train distribution:")
print(train_df["label"].value_counts())

print("\n[+] Test distribution:")
print(test_df["label"].value_counts())

# =========================
# Feature selection
# =========================
FEATURES = [
    "dest_port", "window_duration",
    "fwd_packet_rate", "fwd_byte_rate",
    "bwd_packet_rate", "bwd_byte_rate",      
    "pkt_len_mean", "pkt_len_std",
    "pkt_len_min", "pkt_len_max",
    "syn_ratio", "fin_ratio", "ack_ratio",
    "rst_ratio", "psh_ratio",               
    "fwd_bwd_ratio", "byte_ratio",          
    "iat_mean", "iat_std", "iat_min"
]

X_train = train_df[FEATURES].values
y_train = (train_df["label"] == "malicious").astype(int).values

X_test = test_df[FEATURES].values
y_test = (test_df["label"] == "malicious").astype(int).values

# =========================
# Train model
# =========================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\n[+] Training model...")
model.fit(X_train, y_train)

# =========================
# Evaluation
# =========================
y_pred = model.predict(X_test)

print("\n[+] Classification Report:")
print(classification_report(y_test, y_pred))

print("\n[+] Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\n[+] Key Metrics:")
print(f"False Positives (benign -> malicious): {fp}")
print(f"False Negatives (missed attacks): {fn}")
print(f"True Positives: {tp}")
print(f"True Negatives: {tn}")

fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print(f"\nFalse Positive Rate (FPR): {fpr:.4f}")
print(f"False Negative Rate (FNR): {fnr:.4f}")

# =========================
# Feature importance
# =========================
print("\n[+] Feature Importances:")
for name, importance in zip(FEATURES, model.feature_importances_):
    print(f"{name:20s} -> {importance:.4f}")

# =========================
# Save model
# =========================
joblib.dump(model, "rf_model.pkl")
joblib.dump(FEATURES, "features.pkl")

print("\n[+] Model and features saved")

[+] Found 3 dataset files
[+] Total rows: 40832

[+] Overall label distribution:
label
malicious    20540
normal       20292
Name: count, dtype: int64

[+] Train distribution:
label
malicious    16432
normal       16233
Name: count, dtype: int64

[+] Test distribution:
label
malicious    4108
normal       4059
Name: count, dtype: int64

[+] Training model...

[+] Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4059
           1       1.00      1.00      1.00      4108

    accuracy                           1.00      8167
   macro avg       1.00      1.00      1.00      8167
weighted avg       1.00      1.00      1.00      8167


[+] Confusion Matrix:
[[4059    0]
 [   0 4108]]

[+] Key Metrics:
False Positives (benign -> malicious): 0
False Negatives (missed attacks): 0
True Positives: 4108
True Negatives: 4059

False Positive Rate (FPR): 0.0000
False Negative Rate (FNR): 0.0000

[+] Feature Importances:
de

In [20]:
print(full_df.duplicated(subset=FEATURES).sum())

19436


In [21]:
train_hash = set(map(tuple, X_train))
test_hash = set(map(tuple, X_test))

print(len(train_hash & test_hash))

6255


In [13]:
# Merge all collected CSVs
files = glob.glob("dataset_*.csv")  # if you saved multiple sessions

if len(files) < 2:
    raise ValueError("You need at least 2 dataset files for proper train/test split")

print(f"[+] Found {len(files)} dataset files")

# Use some files for training, others for testing
train_files = files[:-1]
test_files = files[-1:]

print(f"[+] Training on: {train_files}")
print(f"[+] Testing on: {test_files}")

train_df = pd.concat([pd.read_csv(f) for f in train_files], ignore_index=True)
test_df  = pd.concat([pd.read_csv(f) for f in test_files], ignore_index=True)

print("\n[+] Train label distribution:")
print(train_df["label"].value_counts())

print("\n[+] Test label distribution:")
print(test_df["label"].value_counts())

# Drop NaNs

train_df.dropna(inplace=True)
test_df.dropna(inplace=True)

FEATURES = [
    "dest_port", "window_duration",
    "fwd_packet_rate", "fwd_byte_rate",
    "pkt_len_mean", "pkt_len_std",
    "pkt_len_min", "pkt_len_max",
    "syn_ratio", "fin_ratio", "ack_ratio"
]

X_train = train_df[FEATURES].values
y_train = (train_df["label"] == "malicious").astype(int).values

X_test = test_df[FEATURES].values
y_test = (test_df["label"] == "malicious").astype(int).values
#df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

model = RandomForestClassifier(
n_estimators=200,
max_depth=10,
min_samples_split=10,
class_weight="balanced",
random_state=42,
n_jobs=-1
)

print("\n[+] Training model...")
model.fit(X_train, y_train)
# Check balance
#print(test_df["label"].value_counts())

y_pred = model.predict(X_test)

print("\n[+] Classification Report:")
print(classification_report(y_test, y_pred))

print("\n[+] Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# =========================

# Feature importance (debugging)

# =========================

print("\n[+] Feature Importances:")
for name, importance in zip(FEATURES, model.feature_importances_):
    print(f"{name:20s} -> {importance:.4f}")

# =========================

# Save model

# =========================

joblib.dump(model, "rf_model.pkl")
joblib.dump(FEATURES, "features.pkl")

print("\n[+] Model and features saved")

[+] Found 2 dataset files
[+] Training on: ['dataset_malicious.csv']
[+] Testing on: ['dataset_normal.csv']

[+] Train label distribution:
label
malicious    10270
Name: count, dtype: int64

[+] Test label distribution:
label
normal    10146
Name: count, dtype: int64

[+] Training model...

[+] Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00   10146.0
           1       0.00      0.00      0.00       0.0

    accuracy                           0.00   10146.0
   macro avg       0.00      0.00      0.00   10146.0
weighted avg       0.00      0.00      0.00   10146.0


[+] Confusion Matrix:
[[    0 10146]
 [    0     0]]

[+] Feature Importances:
dest_port            -> 0.0000
window_duration      -> 0.0000
fwd_packet_rate      -> 0.0000
fwd_byte_rate        -> 0.0000
pkt_len_mean         -> 0.0000
pkt_len_std          -> 0.0000
pkt_len_min          -> 0.0000
pkt_len_max          -> 0.0000
syn_ratio            -> 0.

c:\Users\izzyd\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\izzyd\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\izzyd\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [14]:
# Merge all collected CSVs
files = glob.glob("dataset_*.csv")  # if you saved multiple sessions
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Check balance
print(df["label"].value_counts())

# Check for NaNs
print(df.isnull().sum())

# Drop any bad rows
df.dropna(inplace=True)

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.to_csv("dataset_final.csv", index=False)
print(f"[+] Final dataset: {len(df)} rows")

label
malicious    10270
normal       10146
Name: count, dtype: int64
dest_port          0
window_duration    0
fwd_packet_rate    0
fwd_byte_rate      0
bwd_packet_rate    0
bwd_byte_rate      0
pkt_len_mean       0
pkt_len_std        0
pkt_len_min        0
pkt_len_max        0
syn_ratio          0
fin_ratio          0
ack_ratio          0
rst_ratio          0
psh_ratio          0
fwd_bwd_ratio      0
byte_ratio         0
iat_mean           0
iat_std            0
iat_min            0
label              0
dtype: int64
[+] Final dataset: 20416 rows


In [15]:
# Check for NaNs
print(df.isnull().sum())

# Drop any bad rows
df.dropna(inplace=True)

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

dest_port          0
window_duration    0
fwd_packet_rate    0
fwd_byte_rate      0
bwd_packet_rate    0
bwd_byte_rate      0
pkt_len_mean       0
pkt_len_std        0
pkt_len_min        0
pkt_len_max        0
syn_ratio          0
fin_ratio          0
ack_ratio          0
rst_ratio          0
psh_ratio          0
fwd_bwd_ratio      0
byte_ratio         0
iat_mean           0
iat_std            0
iat_min            0
label              0
dtype: int64


In [16]:
df.to_csv("dataset_final.csv", index=False)

print(f"[+] Final dataset: {len(df)} rows")

[+] Final dataset: 20416 rows


In [17]:
df = pd.read_csv("dataset_final.csv")

FEATURES = [
    "dest_port", "window_duration",
    "fwd_packet_rate", "fwd_byte_rate",
    "pkt_len_mean", "pkt_len_std",
    "pkt_len_min", "pkt_len_max",
    "syn_ratio", "fin_ratio", "ack_ratio"
]

In [18]:
print("=== Class Distribution ===")
print(df["label"].value_counts())
print(f"\nRatio: {df['label'].value_counts()['benign'] / df['label'].value_counts()['malicious']:.2f}x more benign than malicious")

print("\n=== Feature Stats by Class ===")
for col in df.columns[:-1]:
    benign_mean    = df[df["label"]=="benign"][col].mean()
    malicious_mean = df[df["label"]=="malicious"][col].mean()
    print(f"{col:20s}  benign={benign_mean:10.4f}  malicious={malicious_mean:10.4f}")

=== Class Distribution ===
label
malicious    10270
normal       10146
Name: count, dtype: int64


KeyError: 'benign'

In [ ]:
X = df[FEATURES].values
y = (df["label"] == "malicious").astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2009
           1       1.00      1.00      1.00      2008

    accuracy                           1.00      4017
   macro avg       1.00      1.00      1.00      4017
weighted avg       1.00      1.00      1.00      4017



In [ ]:
# Save model and feature list
joblib.dump(model, "rf_model.pkl")
joblib.dump(FEATURES, "features.pkl")
print("[+] Model and features saved")

[+] Model and features saved


In [ ]:
importance_df = pd.DataFrame({
    "Feature":    FEATURES,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print("=== Feature Importance ===")
print(importance_df.to_string())

# Probability distribution — are predictions clustered near 0.5?
probs = model.predict_proba(X)[:, 1]
print(f"\n=== Probability Distribution ===")
print(f"Mean prob (benign windows):    {probs[y==0].mean():.4f}")
print(f"Mean prob (malicious windows): {probs[y==1].mean():.4f}")
print(f"Benign windows predicted >0.7: {(probs[y==0] > 0.7).sum()} / {(y==0).sum()}")

=== Feature Importance ===
            Feature  Importance
7       pkt_len_max    0.413822
5       pkt_len_std    0.095334
2   fwd_packet_rate    0.092454
3     fwd_byte_rate    0.077980
10        ack_ratio    0.071038
4      pkt_len_mean    0.069128
8         syn_ratio    0.060672
9         fin_ratio    0.053827
1   window_duration    0.037614
6       pkt_len_min    0.028131
0         dest_port    0.000000

=== Probability Distribution ===
Mean prob (benign windows):    0.0011
Mean prob (malicious windows): 0.9990
Benign windows predicted >0.7: 0 / 10042


In [ ]:
# After fitting the model, check false positive rate specifically
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Benign", "Malicious"]))

# False positive analysis — benign windows predicted as malicious
fp_mask = (y_test == 0) & (y_pred == 1)
print(f"\nFalse positives: {fp_mask.sum()} / {(y_test==0).sum()} benign windows")
print("\nFalse positive feature averages:")
fp_features = pd.DataFrame(X_test[fp_mask], columns=FEATURES)
print(fp_features.mean().to_string())

# What do these false positives look like?
# This tells you exactly which benign traffic looks malicious to the model

              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00      2009
   Malicious       1.00      1.00      1.00      2008

    accuracy                           1.00      4017
   macro avg       1.00      1.00      1.00      4017
weighted avg       1.00      1.00      1.00      4017


False positives: 0 / 2009 benign windows

False positive feature averages:
dest_port         NaN
window_duration   NaN
fwd_packet_rate   NaN
fwd_byte_rate     NaN
pkt_len_mean      NaN
pkt_len_std       NaN
pkt_len_min       NaN
pkt_len_max       NaN
syn_ratio         NaN
fin_ratio         NaN
ack_ratio         NaN


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Test with increasing tree depths to see where overfitting starts
print("=== Depth vs Accuracy (overfitting check) ===")
for depth in [3, 5, 10, 20, None]:
    rf_test = RandomForestClassifier(
        n_estimators=50, 
        max_depth=depth, 
        random_state=42
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(rf_test, X, y, cv=cv, scoring="f1")
    print(f"  max_depth={str(depth):>5s}  CV f1={scores.mean():.4f} (+/-{scores.std():.4f})")

# Check if training accuracy vs test accuracy diverge — 
# large gap = overfitting
model.fit(X_train, y_train)
train_acc = model.score(X_train, y_train)
test_acc  = model.score(X_test, y_test)
print(f"\nTrain accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Gap:            {train_acc - test_acc:.4f}  (>0.02 suggests overfitting)")

=== Depth vs Accuracy (overfitting check) ===
  max_depth=    3  CV f1=0.9931 (+/-0.0004)
  max_depth=    5  CV f1=0.9998 (+/-0.0002)
  max_depth=   10  CV f1=1.0000 (+/-0.0000)
  max_depth=   20  CV f1=1.0000 (+/-0.0000)
  max_depth= None  CV f1=1.0000 (+/-0.0000)

Train accuracy: 1.0000
Test accuracy:  1.0000
Gap:            0.0000  (>0.02 suggests overfitting)


In [ ]:
import joblib
import numpy as np

model    = joblib.load("rf_model.pkl")
features = joblib.load("features.pkl")

# Paste your actual live debug values here
live_benign_sample = np.array([[
    80,      # dest_port
    2.1,     # window_duration
    45.0,    # fwd_packet_rate
    3200.0,  # fwd_byte_rate
    210.0,   # pkt_len_mean
    280.0,   # pkt_len_std
    66.0,    # pkt_len_min
    1200.0,  # pkt_len_max
    0.12,    # syn_ratio
    0.15,    # fin_ratio
    0.88,    # ack_ratio
]])

prob = model.predict_proba(live_benign_sample)[0][1]
print(f"Probability of malicious: {prob:.4f}")
print(f"Prediction: {'MALICIOUS' if prob > 0.7 else 'BENIGN'}")

# Now check where this sample sits relative to training data
import pandas as pd
df = pd.read_csv("dataset_final.csv")
print("\nHow does this compare to training data?")
for i, feat in enumerate(features):
    val = live_benign_sample[0][i]
    ben_mean = df[df["label"]=="benign"][feat].mean()
    ben_std  = df[df["label"]=="benign"][feat].std()
    z_score  = (val - ben_mean) / (ben_std + 1e-9)
    print(f"  {feat:20s} live={val:10.2f}  "
          f"train_mean={ben_mean:10.2f}  z={z_score:+.2f}")

Probability of malicious: 0.5500
Prediction: BENIGN

How does this compare to training data?
  dest_port            live=     80.00  train_mean=     80.00  z=+0.00
  window_duration      live=      2.10  train_mean=      1.68  z=+0.04
  fwd_packet_rate      live=     45.00  train_mean=     82.78  z=-0.32
  fwd_byte_rate        live=   3200.00  train_mean=   6642.98  z=-0.33
  pkt_len_mean         live=    210.00  train_mean=    201.51  z=+0.15
  pkt_len_std          live=    280.00  train_mean=    291.07  z=-0.21
  pkt_len_min          live=     66.00  train_mean=     66.02  z=-0.05
  pkt_len_max          live=   1200.00  train_mean=   1068.16  z=+0.37
  syn_ratio            live=      0.12  train_mean=      0.16  z=-0.42
  fin_ratio            live=      0.15  train_mean=      0.18  z=-1.49
  ack_ratio            live=      0.88  train_mean=      0.92  z=-0.54
